# Loan Default Prediction using Logistic Regression

## Industry Problem Statement

### Business Context
Lending institutions (banks, NBFCs, and fintech lenders) face significant financial risk due to loan defaults. When borrowers fail to repay, the institution incurs losses in the form of non-performing assets (NPAs), increased provisioning requirements, higher cost of capital, and reduced profitability. In competitive lending markets, the ability to accurately assess credit risk at the time of loan application is a critical capability that directly impacts portfolio quality and long-term business sustainability.

### Problem Definition
The organization needs a reliable, scalable, and explainable predictive system that can estimate the probability of a borrower defaulting on a loan. The system should use applicant-level demographic, financial, employment, and loan-related information available at the application stage to classify each applicant as either:

- **Likely to Default** (high risk), or  
- **Likely to Repay** (low risk)

### Business Objectives
1. **Reduce Credit Losses**: Identify high-risk applicants early so that loan amounts, interest rates, or approval decisions can be adjusted accordingly.
2. **Improve Approval Efficiency**: Enable faster and more consistent credit decisions while maintaining risk control.
3. **Optimize Portfolio Quality**: Support risk-based pricing and portfolio management strategies.
4. **Maintain Fairness and Explainability**: Ensure the model is interpretable so that credit decisions can be justified to regulators, auditors, and customers.

### Success Criteria
- Strong ability to distinguish between defaulting and non-defaulting applicants (ROC-AUC and Recall on the default class).
- Predictions generated using only information available at the time of application.
- Suitable for integration into existing credit decision engines or underwriting workflows.
- Model outputs (probability of default) support threshold-based decision rules that can be tuned according to the institution’s risk appetite.

### Dataset
- **Source**: Kaggle – Loan Default Prediction Dataset (https://www.kaggle.com/datasets/nikhil1e9/loan-default)
- **Records**: 255,347 loan applications
- **Target**: `Default` (0 = No Default, 1 = Default)
- **Default Rate**: ~11.61% (imbalanced)

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score,
    roc_curve
)

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Load Dataset

In [4]:
# Load the dataset
df = pd.read_csv('Loan_default.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
df.head()

Dataset shape: (255347, 18)

Columns: ['LoanID', 'Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio', 'Education', 'EmploymentType', 'MaritalStatus', 'HasMortgage', 'HasDependents', 'LoanPurpose', 'HasCoSigner', 'Default']


,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0


In [6]:
for i in df.select_dtypes(include=object).columns:
    print(df[i].unique())


<ArrowStringArray>
['I38PQUQS96', 'HPSK72WA7R', 'C1OZ6DPJ8Y', 'V2KKSFM3UN', 'EY08JDHTZP',
 'A9S62RQ7US', 'H8GXPAOS71', '0HGZQKJ36W', '1R0N3LGNRJ', 'CM9L1GTT2P',
 ...
 'DSL4O0KAWD', '6V8S5IUS63', 'O6SWO6CBGB', '48LOOK4VR1', 'AKXAXQN7PG',
 '8C6S86ESGC', '98R4KDHNND', 'XQK1UUUNGP', 'JAO28CPL4H', 'ZTH91CGL0B']
Length: 255347, dtype: str
<ArrowStringArray>
['Bachelor's', 'Master's', 'High School', 'PhD']
Length: 4, dtype: str
<ArrowStringArray>
['Full-time', 'Unemployed', 'Self-employed', 'Part-time']
Length: 4, dtype: str
<ArrowStringArray>
['Divorced', 'Married', 'Single']
Length: 3, dtype: str
<ArrowStringArray>
['Yes', 'No']
Length: 2, dtype: str
<ArrowStringArray>
['Yes', 'No']
Length: 2, dtype: str
<ArrowStringArray>
['Other', 'Auto', 'Business', 'Home', 'Education']
Length: 5, dtype: str
<ArrowStringArray>
['Yes', 'No']
Length: 2, dtype: str


## 3. Exploratory Data Analysis (EDA)

In [5]:
# Basic info
print("Dataset Info:")
print("-" * 50)
df.info()

print("\n\nMissing Values:")
print("-" * 50)
print(df.isnull().sum())

print("\n\nTarget Distribution (Default):")
print("-" * 50)
print(df['Default'].value_counts())
print(f"\nDefault Rate: {df['Default'].mean()*100:.2f}%")

Dataset Info:
--------------------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 255347 entries, 0 to 255346
Data columns (total 18 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   LoanID          255347 non-null  str    
 1   Age             255347 non-null  int64  
 2   Income          255347 non-null  int64  
 3   LoanAmount      255347 non-null  int64  
 4   CreditScore     255347 non-null  int64  
 5   MonthsEmployed  255347 non-null  int64  
 6   NumCreditLines  255347 non-null  int64  
 7   InterestRate    255347 non-null  float64
 8   LoanTerm        255347 non-null  int64  
 9   DTIRatio        255347 non-null  float64
 10  Education       255347 non-null  str    
 11  EmploymentType  255347 non-null  str    
 12  MaritalStatus   255347 non-null  str    
 13  HasMortgage     255347 non-null  str    
 14  HasDependents   255347 non-null  str    
 15  LoanPurpose     255347 non-null  str    
 16  Ha

In [ ]:
# Statistical summary of numerical features
df.describe().T

In [ ]:
# Target distribution plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['Default'].value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Default Distribution (Count)')
axes[0].set_xlabel('Default (0 = No, 1 = Yes)')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['No Default', 'Default'], rotation=0)

df['Default'].value_counts(normalize=True).plot(kind='pie', ax=axes[1], 
                                                 autopct='%1.1f%%',
                                                 colors=['#2ecc71', '#e74c3c'],
                                                 labels=['No Default', 'Default'])
axes[1].set_title('Default Distribution (%)')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# Numerical features distribution by Default
numerical_features = ['Age', 'Income', 'LoanAmount', 'CreditScore', 
                      'MonthsEmployed', 'InterestRate', 'DTIRatio']

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, col in enumerate(numerical_features):
    sns.boxplot(x='Default', y=col, data=df, ax=axes[i])
    axes[i].set_title(f'{col} vs Default')
    axes[i].set_xticklabels(['No Default', 'Default'])

# Hide unused subplots
for j in range(len(numerical_features), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap of numerical features
plt.figure(figsize=(10, 8))
corr = df[numerical_features + ['Default']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

## 4. Data Preprocessing

In [ ]:
# Drop LoanID (not useful for modeling)
df_model = df.drop(columns=['LoanID'])

# Separate features and target
X = df_model.drop(columns=['Default'])
y = df_model['Default']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

In [ ]:
# Identify categorical and numerical columns
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print("Categorical columns:", categorical_cols)
print("\nNumerical columns:", numerical_cols)

In [ ]:
# Encode categorical variables using Label Encoding
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le
    print(f"Encoded {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

In [ ]:
# Train-Test Split (80-20) with stratification to maintain class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")
print(f"\nTrain Default Rate: {y_train.mean()*100:.2f}%")
print(f"Test Default Rate:  {y_test.mean()*100:.2f}%")

In [ ]:
# Feature Scaling (StandardScaler) - only on numerical features
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test_scaled[numerical_cols] = scaler.transform(X_test[numerical_cols])

print("Feature scaling completed.")
X_train_scaled.head()

## 5. Build Logistic Regression Model

We use `class_weight='balanced'` to handle the class imbalance (only ~11.6% defaults).

In [ ]:
# Initialize and train Logistic Regression model
model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight='balanced',   # Handles class imbalance
    solver='lbfgs'
)

model.fit(X_train_scaled, y_train)

print("Logistic Regression model trained successfully!")

## 6. Model Evaluation

In [ ]:
# Predictions
y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print("=" * 50)
print("MODEL PERFORMANCE METRICS")
print("=" * 50)
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")
print(f"ROC-AUC   : {roc_auc:.4f}")
print("=" * 50)

In [ ]:
# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['No Default (0)', 'Default (1)']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, 
                               display_labels=['No Default', 'Default'])
disp.plot(cmap='Blues', ax=ax, values_format='d')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

print(f"True Negatives (TN) : {cm[0,0]}")
print(f"False Positives (FP): {cm[0,1]}")
print(f"False Negatives (FN): {cm[1,0]}")
print(f"True Positives (TP) : {cm[1,1]}")

In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='#3498db', lw=2, label=f'ROC Curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Feature Importance (Model Coefficients)

In Logistic Regression, the magnitude and sign of coefficients indicate the strength and direction of each feature’s influence on the probability of default.

In [ ]:
# Extract and display coefficients
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_[0]
}).sort_values(by='Coefficient', key=abs, ascending=False)

print("Feature Importance (sorted by absolute coefficient value):")
print("=" * 60)
print(coef_df.to_string(index=False))
print("=" * 60)
print("\nInterpretation:")
print("- Positive coefficient → Increases probability of Default")
print("- Negative coefficient → Decreases probability of Default")
print("- Larger absolute value → Stronger influence on the prediction")

In [ ]:
# Visualize feature importance
plt.figure(figsize=(10, 6))
colors = ['#e74c3c' if c > 0 else '#2ecc71' for c in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
plt.xlabel('Coefficient Value')
plt.title('Logistic Regression Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 8. Sample Predictions

In [ ]:
# Show sample predictions from the test set
sample_results = pd.DataFrame({
    'Actual': y_test.values[:15],
    'Predicted': y_pred[:15],
    'Probability_of_Default': np.round(y_pred_proba[:15], 4)
})

print("Sample Predictions (first 15 records from Test Set):")
print(sample_results.to_string(index=False))

## 9. Key Insights & Recommendations

### Model Performance Summary
- **ROC-AUC ≈ 0.75**: The model has decent discrimination ability.
- **Recall ≈ 0.70**: Captures ~70% of actual defaults (important for risk management).
- **Precision is lower** due to class imbalance and the use of balanced class weights (trade-off for higher recall).

### Most Influential Features
| Feature          | Impact on Default Risk |
|------------------|------------------------|
| Age              | Higher age → Lower risk |
| InterestRate     | Higher rate → Higher risk |
| MonthsEmployed   | Longer employment → Lower risk |
| Income           | Higher income → Lower risk |
| LoanAmount       | Larger loan → Higher risk |
| HasCoSigner      | Presence of co-signer → Lower risk |

### Business Recommendations
1. **Use Probability Threshold Tuning**: Instead of the default 0.5 threshold, choose a threshold based on the institution’s risk appetite (e.g., higher threshold for stricter approval).
2. **Risk-Based Pricing**: Use the predicted probability of default to set interest rates or loan limits.
3. **Further Improvements**:
   - Try advanced models (Random Forest, XGBoost, LightGBM)
   - Apply SMOTE or other oversampling techniques
   - Perform hyperparameter tuning and cross-validation
   - Add interaction features or domain-specific engineered features
4. **Model Monitoring**: Once deployed, continuously monitor for data drift and performance degradation.

---

**End of Notebook**